<a href="https://colab.research.google.com/github/yox19/Bioinformatics/blob/main/Medication_Reconciliation_%26_Drug_Interaction_Checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### `data/drugs.py`

This file defines the `DrugInfo` dataclass and the `DRUGS` dictionary, which stores all drug definitions. Each entry includes details like the drug's name, category, dosage, renal safety thresholds, and pregnancy safety.

In [ ]:
import dataclasses

@dataclasses.dataclass
class DrugInfo:
    name: str
    category: str
    dose_range: str
    renal_ok: float | None = None
    renal_ci: float | None = None
    pregnancy: str | None = None  # "safe" | "caution" | "contraindicated" | "unknown"
    notes: str | None = None

# Define categories
CATEGORY_ANTIDIABETIC = "Antidiabetic"
CATEGORY_CARDIOVASCULAR = "Cardiovascular"
CATEGORY_ANTICOAGULANT = "Anticoagulant"
CATEGORY_ANTIINFECTIVE = "Anti-infective"

DRUGS = {
    "metformin": DrugInfo(
        name="Metformin",
        category=CATEGORY_ANTIDIABETIC,
        dose_range="500-2000 mg/day",
        renal_ok=45,  # Caution if eGFR 30-45
        renal_ci=30,  # Contraindicated if eGFR <30
        pregnancy="unknown",
        notes="First-line for T2DM. Risk of lactic acidosis with renal impairment.",
    ),
    "enalapril": DrugInfo(
        name="Enalapril",
        category=CATEGORY_CARDIOVASCULAR,
        dose_range="5-40 mg/day",
        renal_ok=30,  # Caution if eGFR 10-30
        renal_ci=10,  # Contraindicated if eGFR <10
        pregnancy="contraindicated",
        notes="ACE inhibitor. Risk of hyperkalaemia and renal dysfunction.",
    ),
    "ibuprofen": DrugInfo(
        name="Ibuprofen",
        category=CATEGORY_ANTIINFECTIVE, # This is an example, should be pain/anti-inflammatory
        dose_range="200-800 mg 3x/day",
        renal_ok=30,
        renal_ci=15,
        pregnancy="caution",
        notes="NSAID. Risk of renal impairment and gastrointestinal upset."
    ),
    "glibenclamide": DrugInfo(
        name="Glibenclamide",
        category=CATEGORY_ANTIDIABETIC,
        dose_range="2.5-20 mg/day",
        renal_ok=50,
        renal_ci=30,
        pregnancy="caution",
        notes="Sulfonylurea. Risk of hypoglycaemia, especially in renal impairment."
    ),
    "warfarin": DrugInfo(
        name="Warfarin",
        category=CATEGORY_ANTICOAGULANT,
        dose_range="2-10 mg/day",
        pregnancy="contraindicated",
        notes="Vitamin K antagonist. Requires regular INR monitoring.",
    ),
    "fluconazole": DrugInfo(
        name="Fluconazole",
        category=CATEGORY_ANTIINFECTIVE,
        dose_range="50-400 mg/day",
        renal_ok=50,
        renal_ci=20,
        pregnancy="caution",
        notes="Antifungal. Potent CYP2C9 inhibitor.",
    ),
    "carvedilol": DrugInfo(
        name="Carvedilol",
        category=CATEGORY_CARDIOVASCULAR,
        dose_range="3.125-25 mg 2x/day",
        pregnancy="unknown",
        notes="Non-selective beta-blocker with alpha-blocking activity.",
    ),
    "spironolactone": DrugInfo(
        name="Spironolactone",
        category=CATEGORY_CARDIOVASCULAR,
        dose_range="25-100 mg/day",
        renal_ok=45,
        renal_ci=30,
        pregnancy="unknown",
        notes="Potassium-sparing diuretic. Risk of hyperkalaemia.",
    ),
    "empagliflozin": DrugInfo(
        name="Empagliflozin",
        category=CATEGORY_ANTIDIABETIC,
        dose_range="10–25 mg/day",
        renal_ok=45,        # Caution if eGFR 30-45
        renal_ci=20,        # Contraindicated if eGFR <20
        pregnancy="contraindicated",
        notes="SGLT2 inhibitor. Risk of DKA, UTI, genital infections. "
              "Also reduces HF hospitalisation and CKD progression.",
    ),
    "gliclazide": DrugInfo(
        name="Gliclazide",
        category=CATEGORY_ANTIDIABETIC,
        dose_range="30-120 mg/day",
        pregnancy="caution",
        notes="Sulfonylurea. Less risk of hypoglycaemia than glibenclamide."
    ),
    "insulin": DrugInfo(
        name="Insulin",
        category=CATEGORY_ANTIDIABETIC,
        dose_range="Individualised",
        pregnancy="safe",
        notes="Essential for T1DM, often used in T2DM."
    )
}

### `data/interactions.py`

This file defines the `InteractionRule` dataclass and the `RULES` list, which contains all the drug interaction and contraindication rules. These rules are central to how MedRec identifies potential safety issues.

In [8]:
import os

# Create the data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Create an empty __init__.py file in the data directory to mark it as a package
with open('data/__init__.py', 'w') as f:
    pass

# Write the content of drugs.py to the file
with open('data/drugs.py', 'w') as f:
    f.write('''import dataclasses\n\n@dataclasses.dataclass\nclass DrugInfo:\n    name: str\n    category: str\n    dose_range: str\n    renal_ok: float | None = None\n    renal_ci: float | None = None\n    pregnancy: str | None = None  # "safe" | "caution" | "contraindicated" | "unknown"\n    notes: str | None = None\n\n# Define categories\nCATEGORY_ANTIDIABETIC = "Antidiabetic"\nCATEGORY_CARDIOVASCULAR = "Cardiovascular"\nCATEGORY_ANTICOAGULANT = "Anticoagulant"\nCATEGORY_ANTIINFECTIVE = "Anti-infective"\n\nDRUGS = {\n    "metformin": DrugInfo(\n        name="Metformin",\n        category=CATEGORY_ANTIDIABETIC,\n        dose_range="500-2000 mg/day",\n        renal_ok=45,  # Caution if eGFR 30-45\n        renal_ci=30,  # Contraindicated if eGFR <30\n        pregnancy="unknown",\n        notes="First-line for T2DM. Risk of lactic acidosis with renal impairment.",\n    ),\n    "enalapril": DrugInfo(\n        name="Enalapril",\n        category=CATEGORY_CARDIOVASCULAR,\n        dose_range="5-40 mg/day",\n        renal_ok=30,  # Caution if eGFR 10-30\n        renal_ci=10,  # Contraindicated if eGFR <10\n        pregnancy="contraindicated",\n        notes="ACE inhibitor. Risk of hyperkalaemia and renal dysfunction.",\n    ),\n    "ibuprofen": DrugInfo(\n        name="Ibuprofen",\n        category=CATEGORY_ANTIINFECTIVE, # This is an example, should be pain/anti-inflammatory\n        dose_range="200-800 mg 3x/day",\n        renal_ok=30,\n        renal_ci=15,\n        pregnancy="caution",\n        notes="NSAID. Risk of renal impairment and gastrointestinal upset."\n    ),\n    "glibenclamide": DrugInfo(\n        name="Glibenclamide",\n        category=CATEGORY_ANTIDIABETIC,\n        dose_range="2.5-20 mg/day",\n        renal_ok=50,\n        renal_ci=30,\n        pregnancy="caution",\n        notes="Sulfonylurea. Risk of hypoglycaemia, especially in renal impairment."\n    ),\n    "warfarin": DrugInfo(\n        name="Warfarin",\n        category=CATEGORY_ANTICOAGULANT,\n        dose_range="2-10 mg/day",\n        pregnancy="contraindicated",\n        notes="Vitamin K antagonist. Requires regular INR monitoring.",\n    ),\n    "fluconazole": DrugInfo(\n        name="Fluconazole",\n        category=CATEGORY_ANTIINFECTIVE,\n        dose_range="50-400 mg/day",\n        renal_ok=50,\n        renal_ci=20,\n        pregnancy="caution",\n        notes="Antifungal. Potent CYP2C9 inhibitor.",\n    ),\n    "carvedilol": DrugInfo(\n        name="Carvedilol",\n        category=CATEGORY_CARDIOVASCULAR,\n        dose_range="3.125-25 mg 2x/day",\n        pregnancy="unknown",\n        notes="Non-selective beta-blocker with alpha-blocking activity.",\n    ),\n    "spironolactone": DrugInfo(\n        name="Spironolactone",\n        category=CATEGORY_CARDIOVASCULAR,\n        dose_range="25-100 mg/day",\n        renal_ok=45,\n        renal_ci=30,\n        pregnancy="unknown",\n        notes="Potassium-sparing diuretic. Risk of hyperkalaemia.",\n    ),\n    "empagliflozin": DrugInfo(\n        name="Empagliflozin",\n        category=CATEGORY_ANTIDIABETIC,\n        dose_range="10–25 mg/day",\n        renal_ok=45,        # Caution if eGFR 30-45\n        renal_ci=20,        # Contraindicated if eGFR <20\n        pregnancy="contraindicated",\n        notes="SGLT2 inhibitor. Risk of DKA, UTI, genital infections. "\n              "Also reduces HF hospitalisation and CKD progression.",\n    ),\n    "gliclazide": DrugInfo(\n        name="Gliclazide",\n        category=CATEGORY_ANTIDIABETIC,\n        dose_range="30-120 mg/day",\n        pregnancy="caution",\n        notes="Sulfonylurea. Less risk of hypoglycaemia than glibenclamide."\n    ),\n    "insulin": DrugInfo(\n        name="Insulin",\n        category=CATEGORY_ANTIDIABETIC,\n        dose_range="Individualised",\n        pregnancy="safe",\n        notes="Essential for T1DM, often used in T2DM."\n    )\n}''')

# Write the content of interactions.py to the file
with open('data/interactions.py', 'w') as f:
    f.write('''import dataclasses\n\n@dataclasses.dataclass\nclass InteractionRule:\n    drug_a: str\n    drug_b: str | None\n    condition: str | None\n    egfr_below: float | None\n    severity: str  # "critical" | "warn" | "info"\n    title: str\n    detail: str\n    action: str\n    references: str | None = None\n\n# Define conditions for type-checking and consistency\nCONDITIONS = {\n    "CKD",\n    "Diabetes (T2DM)",\n    "Pregnancy",\n    "Asthma/COPD",\n    "Heart Failure"\n}\n\nRULES = [\n    # Type A — Drug-Drug Interaction (drug_a + drug_b):\n    InteractionRule(\n        drug_a="warfarin",\n        drug_b="fluconazole",\n        condition=None,\n        egfr_below=None,\n        severity="critical",\n        title="Warfarin + fluconazole — severe INR elevation",\n        detail="Fluconazole potently inhibits CYP2C9, dramatically raising warfarin levels and increasing bleeding risk.",\n        action="Avoid combination. If essential, reduce warfarin by 50% and monitor INR daily.",\n        references="BNF; Stockley's Drug Interactions",\n    ),\n    InteractionRule(\n        drug_a="enalapril",\n        drug_b="spironolactone",\n        condition=None,\n        egfr_below=None,\n        severity="warn",\n        title="ACEi + Spironolactone — risk of hyperkalaemia",\n        detail="Both drugs increase potassium. Combination can lead to dangerous hyperkalaemia, especially in renal impairment.",\n        action="Monitor serum potassium closely, especially when initiating or titrating doses. Avoid if eGFR < 30.",\n        references="BNF",\n    ),\n    # Type B — Condition-Drug Rule (condition only, no drug_b):\n    InteractionRule(\n        drug_a="carvedilol",\n        drug_b=None,\n        condition="Asthma/COPD",\n        egfr_below=None,\n        severity="critical",\n        title="Non-selective beta-blocker contraindicated in asthma",\n        detail="Carvedilol blocks beta-2 receptors, which can cause life-threatening bronchospasm in patients with asthma or severe COPD.",\n        action="Contraindicated in asthma/severe COPD. Use a cardioselective beta-blocker (e.g., bisoprolol) only if compelling cardiac indication.",\n        references="BNF",\n    ),\n    InteractionRule(\n        drug_a="glibenclamide",\n        drug_b=None,\n        condition="CKD",\n        egfr_below=None, # This rule is covered by renal pass, but explicit rule can add detail\n        severity="critical",\n        title="Glibenclamide contraindicated in CKD",\n        detail="Glibenclamide has active metabolites renally excreted, leading to prolonged severe hypoglycaemia in CKD patients.",\n        action="Stop glibenclamide. Switch to an alternative antidiabetic agent such as gliclazide (with caution) or insulin.",\n        references="KDIGO 2022",\n    ),\n    InteractionRule(\n        drug_a="enalapril",\n        drug_b=None,\n        condition="Pregnancy",\n        egfr_below=None,\n        severity="critical",\n        title="ACE inhibitors contraindicated in pregnancy",\n        detail="ACE inhibitors can cause foetal renal dysfunction, oligohydramnios, and skeletal malformations, particularly in the second and third trimesters.",\n        action="Stop enalapril immediately. Switch to alternative antihypertensives like methyldopa or labetalol.",\n        references="FDA Black Box Warning",\n    ),\n    # Type C — Renal threshold rule (drug_a + egfr_below):\n    InteractionRule(\n        drug_a="metformin",\n        drug_b=None,\n        condition=None,\n        egfr_below=30,\n        severity="critical",\n        title="Metformin contraindicated: eGFR < 30",\n        detail="Metformin accumulates in severe renal impairment, significantly increasing the risk of lactic acidosis, which can be fatal.",\n        action="Stop metformin immediately. Switch to insulin or gliclazide (with caution).",\n        references="KDIGO 2022",\n    ),\n    # Type D — Combined: drug-drug + renal (all three):\n    InteractionRule(\n        drug_a="spironolactone",\n        drug_b="enalapril",\n        condition="CKD",\n        egfr_below=30,\n        severity="critical",\n        title="Spiro + ACEi in severe CKD — extreme hyperkalaemia",\n        detail="The combination of spironolactone and an ACE inhibitor significantly increases the risk of severe hyperkalaemia in patients with chronic kidney disease (eGFR < 30) due to impaired renal potassium excretion.",\n        action="Contraindicated. Stop spironolactone. Arrange urgent nephrology review and consider alternative blood pressure management.",\n        references="BNF; Clinical guidelines",\n    ),\n    InteractionRule(\n        drug_a="ibuprofen",\n        drug_b="enalapril",\n        condition=None,\n        egfr_below=60,\n        severity="warn",\n        title="Triple Whammy: NSAID + ACEi in renal impairment",\n        detail="Combination of Ibuprofen (NSAID) and Enalapril (ACEi) can cause acute kidney injury, especially with reduced eGFR, by affecting renal hemodynamics.",\n        action="Avoid combination, especially if eGFR < 60. Consider paracetamol instead of ibuprofen. Monitor renal function closely if unavoidable.",\n        references="European Medicines Agency",\n    )\n]''')


In [ ]:
import dataclasses

@dataclasses.dataclass
class InteractionRule:
    drug_a: str
    drug_b: str | None
    condition: str | None
    egfr_below: float | None
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    references: str | None = None

# Define conditions for type-checking and consistency
CONDITIONS = {
    "CKD",
    "Diabetes (T2DM)",
    "Pregnancy",
    "Asthma/COPD",
    "Heart Failure"
}

RULES = [
    # Type A — Drug-Drug Interaction (drug_a + drug_b):
    InteractionRule(
        drug_a="warfarin",
        drug_b="fluconazole",
        condition=None,
        egfr_below=None,
        severity="critical",
        title="Warfarin + fluconazole — severe INR elevation",
        detail="Fluconazole potently inhibits CYP2C9, dramatically raising warfarin levels and increasing bleeding risk.",
        action="Avoid combination. If essential, reduce warfarin by 50% and monitor INR daily.",
        references="BNF; Stockley's Drug Interactions",
    ),
    InteractionRule(
        drug_a="enalapril",
        drug_b="spironolactone",
        condition=None,
        egfr_below=None,
        severity="warn",
        title="ACEi + Spironolactone — risk of hyperkalaemia",
        detail="Both drugs increase potassium. Combination can lead to dangerous hyperkalaemia, especially in renal impairment.",
        action="Monitor serum potassium closely, especially when initiating or titrating doses. Avoid if eGFR < 30.",
        references="BNF",
    ),
    # Type B — Condition-Drug Rule (condition only, no drug_b):
    InteractionRule(
        drug_a="carvedilol",
        drug_b=None,
        condition="Asthma/COPD",
        egfr_below=None,
        severity="critical",
        title="Non-selective beta-blocker contraindicated in asthma",
        detail="Carvedilol blocks beta-2 receptors, which can cause life-threatening bronchospasm in patients with asthma or severe COPD.",
        action="Contraindicated in asthma/severe COPD. Use a cardioselective beta-blocker (e.g., bisoprolol) only if compelling cardiac indication.",
        references="BNF",
    ),
    InteractionRule(
        drug_a="glibenclamide",
        drug_b=None,
        condition="CKD",
        egfr_below=None, # This rule is covered by renal pass, but explicit rule can add detail
        severity="critical",
        title="Glibenclamide contraindicated in CKD",
        detail="Glibenclamide has active metabolites renally excreted, leading to prolonged severe hypoglycaemia in CKD patients.",
        action="Stop glibenclamide. Switch to an alternative antidiabetic agent such as gliclazide (with caution) or insulin.",
        references="KDIGO 2022",
    ),
    InteractionRule(
        drug_a="enalapril",
        drug_b=None,
        condition="Pregnancy",
        egfr_below=None,
        severity="critical",
        title="ACE inhibitors contraindicated in pregnancy",
        detail="ACE inhibitors can cause foetal renal dysfunction, oligohydramnios, and skeletal malformations, particularly in the second and third trimesters.",
        action="Stop enalapril immediately. Switch to alternative antihypertensives like methyldopa or labetalol.",
        references="FDA Black Box Warning",
    ),
    # Type C — Renal threshold rule (drug_a + egfr_below):
    InteractionRule(
        drug_a="metformin",
        drug_b=None,
        condition=None,
        egfr_below=30,
        severity="critical",
        title="Metformin contraindicated: eGFR < 30",
        detail="Metformin accumulates in severe renal impairment, significantly increasing the risk of lactic acidosis, which can be fatal.",
        action="Stop metformin immediately. Switch to insulin or gliclazide (with caution).",
        references="KDIGO 2022",
    ),
    # Type D — Combined: drug-drug + renal (all three):
    InteractionRule(
        drug_a="spironolactone",
        drug_b="enalapril",
        condition="CKD",
        egfr_below=30,
        severity="critical",
        title="Spiro + ACEi in severe CKD — extreme hyperkalaemia",
        detail="The combination of spironolactone and an ACE inhibitor significantly increases the risk of severe hyperkalaemia in patients with chronic kidney disease (eGFR < 30) due to impaired renal potassium excretion.",
        action="Contraindicated. Stop spironolactone. Arrange urgent nephrology review and consider alternative blood pressure management.",
        references="BNF; Clinical guidelines",
    ),
    InteractionRule(
        drug_a="ibuprofen",
        drug_b="enalapril",
        condition=None,
        egfr_below=60,
        severity="warn",
        title="Triple Whammy: NSAID + ACEi in renal impairment",
        detail="Combination of Ibuprofen (NSAID) and Enalapril (ACEi) can cause acute kidney injury, especially with reduced eGFR, by affecting renal hemodynamics.",
        action="Avoid combination, especially if eGFR < 60. Consider paracetamol instead of ibuprofen. Monitor renal function closely if unavoidable.",
        references="European Medicines Agency",
    )
]


### `core/patient.py`

This file defines the `PatientContext` dataclass, which encapsulates all necessary patient information (demographics, conditions, current medications, etc.) that the `MedRecEngine` uses for its evaluations.

In [ ]:
import dataclasses
from typing import List, Optional

@dataclasses.dataclass(frozen=True)
class PatientContext:
    age: int
    sex: str  # 'M' or 'F'
    egfr: Optional[float] = None  # Estimated Glomerular Filtration Rate
    conditions: List[str] = dataclasses.field(default_factory=list)
    current_meds: List[str] = dataclasses.field(default_factory=list)
    new_meds: List[str] = dataclasses.field(default_factory=list)

    @property
    def all_meds(self) -> List[str]:
        """Returns a combined list of all current and new medications."""
        return sorted(list(set(self.current_meds + self.new_meds)))

### `core/engine.py`

This file implements the `MedRecEngine`, which performs the core drug interaction and contraindication checking. It uses a three-pass evaluation strategy, combining rule-based checks, per-drug renal safety, and per-drug pregnancy safety. It also defines the `CheckResult` dataclass to structure the output of these checks.

In [ ]:
import dataclasses
from typing import List, Optional, Set, Dict
from data.drugs import DRUGS, DrugInfo
from data.interactions import RULES, InteractionRule, CONDITIONS
from core.patient import PatientContext

@dataclasses.dataclass(frozen=True)
class CheckResult:
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    drug_a: str
    drug_a_name: str
    drug_b: Optional[str] = None
    drug_b_name: Optional[str] = None
    condition_triggered: Optional[str] = None
    egfr_triggered: Optional[float] = None
    rule_type: str = "interaction" # 'interaction', 'renal_ci', 'renal_ok', 'pregnancy_ci', 'pregnancy_warn'
    references: Optional[str] = None

    def __lt__(self, other): # For sorting results by severity
        severity_order = {"critical": 0, "warn": 1, "info": 2}
        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)


class MedRecEngine:

    def check(self, ctx: PatientContext) -> List[CheckResult]:
        results: Dict[str, CheckResult] = {}

        all_meds_ids = ctx.all_meds
        all_meds_info = {drug_id: DRUGS[drug_id] for drug_id in all_meds_ids if drug_id in DRUGS}

        # --- Pass 1: Rule-based interaction check ---
        for rule in RULES:
            if rule.drug_a not in all_meds_ids:
                continue

            drug_a_info = DRUGS.get(rule.drug_a)
            if not drug_a_info:
                continue # Should not happen if data is clean

            # Check drug_b presence if required
            if rule.drug_b and rule.drug_b not in all_meds_ids:
                continue
            drug_b_info = DRUGS.get(rule.drug_b) if rule.drug_b else None

            # Check condition if required
            if rule.condition and rule.condition not in ctx.conditions:
                continue

            # Check eGFR if required
            if rule.egfr_below is not None:
                if ctx.egfr is None or ctx.egfr >= rule.egfr_below:
                    continue

            # If all conditions met, add result (deduplicated by title)
            if rule.title not in results:
                results[rule.title] = CheckResult(
                    severity=rule.severity,
                    title=rule.title,
                    detail=rule.detail,
                    action=rule.action,
                    drug_a=rule.drug_a,
                    drug_a_name=drug_a_info.name,
                    drug_b=rule.drug_b,
                    drug_b_name=drug_b_info.name if drug_b_info else None,
                    condition_triggered=rule.condition,
                    egfr_triggered=rule.egfr_below,
                    rule_type="interaction",
                    references=rule.references
                )

        # --- Pass 2: Per-drug renal safety backstop ---
        if ctx.egfr is not None:
            for drug_id, drug_info in all_meds_info.items():
                # Prioritize explicit rules, skip if a rule with the same title already fired
                # This is a simplification; a more robust approach might be needed for complex overlaps

                if drug_info.renal_ci is not None and ctx.egfr < drug_info.renal_ci:
                    title = f"{drug_info.name} contraindicated: eGFR < {int(drug_info.renal_ci)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated for eGFR below {int(drug_info.renal_ci)}. Risk of accumulation and toxicity.",
                            action=f"Stop {drug_info.name}. Consider alternative therapy.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ci,
                            rule_type="renal_ci"
                        )

                elif drug_info.renal_ok is not None and ctx.egfr < drug_info.renal_ok:
                    title = f"{drug_info.name} caution: eGFR < {int(drug_info.renal_ok)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution for eGFR below {int(drug_info.renal_ok)}. Consider dose reduction or increased monitoring.",
                            action=f"Monitor renal function and drug levels. Consider dose adjustment.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ok,
                            rule_type="renal_ok"
                        )

        # --- Pass 3: Per-drug pregnancy safety ---
        if "Pregnancy" in ctx.conditions:
            for drug_id, drug_info in all_meds_info.items():
                # Explicit rules take priority for pregnancy too
                # Check if any rule-based result for this drug already covers pregnancy
                # This is a simplification, more specific checking might be needed

                if drug_info.pregnancy == "contraindicated":
                    title = f"{drug_info.name} contraindicated in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated during pregnancy due to potential foetal harm.",
                            action=f"Stop {drug_info.name} immediately. Discuss alternatives with patient.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_ci"
                        )
                elif drug_info.pregnancy == "caution":
                    title = f"{drug_info.name} caution in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution during pregnancy. Weigh benefits against potential risks.",
                            action=f"Review necessity of {drug_info.name}. Consider closer monitoring or alternative if possible.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_warn"
                        )

        # Convert dict values to list and sort by severity
        sorted_results = sorted(list(results.values()))
        return sorted_results

### `cli.py`

This file provides a command-line interface for the MedRec system. It allows users to input patient data and medications via arguments and get drug interaction and contraindication results.

In [9]:
import os

# Ensure core directory exists
os.makedirs('core', exist_ok=True)

# Create an empty __init__.py file in the core directory to mark it as a package
with open('core/__init__.py', 'w') as f:
    pass

# Write the content of engine.py to the file
with open('core/engine.py', 'w') as f:
    f.write('''import dataclasses\nfrom typing import List, Optional, Set, Dict\nfrom data.drugs import DRUGS, DrugInfo\nfrom data.interactions import RULES, InteractionRule, CONDITIONS\nfrom core.patient import PatientContext\n\n@dataclasses.dataclass(frozen=True)\nclass CheckResult:\n    severity: str  # "critical" | "warn" | "info"\n    title: str\n    detail: str\n    action: str\n    drug_a: str\n    drug_a_name: str\n    drug_b: Optional[str] = None\n    drug_b_name: Optional[str] = None\n    condition_triggered: Optional[str] = None\n    egfr_triggered: Optional[float] = None\n    rule_type: str = "interaction" # 'interaction', 'renal_ci', 'renal_ok', 'pregnancy_ci', 'pregnancy_warn'\n    references: Optional[str] = None\n\n    def __lt__(self, other): # For sorting results by severity\n        severity_order = {"critical": 0, "warn": 1, "info": 2}\n        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)\n\n\nclass MedRecEngine:\n\n    def check(self, ctx: PatientContext) -> List[CheckResult]:\n        results: Dict[str, CheckResult] = {}\n\n        all_meds_ids = ctx.all_meds\n        all_meds_info = {drug_id: DRUGS[drug_id] for drug_id in all_meds_ids if drug_id in DRUGS}\n\n        # --- Pass 1: Rule-based interaction check ---\n        for rule in RULES:\n            if rule.drug_a not in all_meds_ids:\n                continue\n\n            drug_a_info = DRUGS.get(rule.drug_a)\n            if not drug_a_info:\n                continue # Should not happen if data is clean\n\n            # Check drug_b presence if required\n            if rule.drug_b and rule.drug_b not in all_meds_ids:\n                continue\n            drug_b_info = DRUGS.get(rule.drug_b) if rule.drug_b else None\n\n            # Check condition if required\n            if rule.condition and rule.condition not in ctx.conditions:\n                continue\n\n            # Check eGFR if required\n            if rule.egfr_below is not None:\n                if ctx.egfr is None or ctx.egfr >= rule.egfr_below:\n                    continue\n\n            # If all conditions met, add result (deduplicated by title)\n            if rule.title not in results:\n                results[rule.title] = CheckResult(\n                    severity=rule.severity,\n                    title=rule.title,\n                    detail=rule.detail,\n                    action=rule.action,\n                    drug_a=rule.drug_a,\n                    drug_a_name=drug_a_info.name,\n                    drug_b=rule.drug_b,\n                    drug_b_name=drug_b_info.name if drug_b_info else None,\n                    condition_triggered=rule.condition,\n                    egfr_triggered=rule.egfr_below,\n                    rule_type="interaction",\n                    references=rule.references\n                )\n\n        # --- Pass 2: Per-drug renal safety backstop ---\n        if ctx.egfr is not None:\n            for drug_id, drug_info in all_meds_info.items():\n                # Prioritize explicit rules, skip if a rule with the same title already fired\n                # This is a simplification; a more robust approach might be needed for complex overlaps\n                \n                if drug_info.renal_ci is not None and ctx.egfr < drug_info.renal_ci:\n                    title = f"{drug_info.name} contraindicated: eGFR < {int(drug_info.renal_ci)}"\n                    if title not in results:\n                        results[title] = CheckResult(\n                            severity="critical",\n                            title=title,\n                            detail=f"{drug_info.name} is contraindicated for eGFR below {int(drug_info.renal_ci)}. Risk of accumulation and toxicity.",\n                            action=f"Stop {drug_id.capitalize()}. Consider alternative therapy.",\n                            drug_a=drug_id,\n                            drug_a_name=drug_info.name,\n                            egfr_triggered=drug_info.renal_ci,\n                            rule_type="renal_ci"\n                        )\n\n                elif drug_info.renal_ok is not None and ctx.egfr < drug_info.renal_ok:\n                    title = f"{drug_info.name} caution: eGFR < {int(drug_info.renal_ok)}"\n                    if title not in results:\n                        results[title] = CheckResult(\n                            severity="warn",\n                            title=title,\n                            detail=f"Use {drug_info.name} with caution for eGFR below {int(drug_info.renal_ok)}. Consider dose reduction or increased monitoring.",\n                            action=f"Monitor renal function and drug levels. Consider dose adjustment.",\n                            drug_a=drug_id,\n                            drug_a_name=drug_info.name,\n                            egfr_triggered=drug_info.renal_ok,\n                            rule_type="renal_ok"\n                        )\n\n        # --- Pass 3: Per-drug pregnancy safety ---\n        if "Pregnancy" in ctx.conditions:\n            for drug_id, drug_info in all_meds_info.items():\n                # Explicit rules take priority for pregnancy too\n                # Check if any rule-based result for this drug already covers pregnancy\n                # This is a simplification, more specific checking might be needed\n\n                if drug_info.pregnancy == "contraindicated":\n                    title = f"{drug_info.name} contraindicated in pregnancy"\n                    if title not in results:\n                        results[title] = CheckResult(\n                            severity="critical",\n                            title=title,\n                            detail=f"{drug_info.name} is contraindicated during pregnancy due to potential foetal harm.",\n                            action=f"Stop {drug_info.name} immediately. Discuss alternatives with patient.",\n                            drug_a=drug_id,\n                            drug_a_name=drug_info.name,\n                            condition_triggered="Pregnancy",\n                            rule_type="pregnancy_ci"\n                        )\n                elif drug_info.pregnancy == "caution":\n                    title = f"{drug_info.name} caution in pregnancy"\n                    if title not in results:\n                        results[title] = CheckResult(\n                            severity="warn",\n                            title=title,\n                            detail=f"Use {drug_info.name} with caution during pregnancy. Weigh benefits against potential risks.",\n                            action=f"Review necessity of {drug_info.name}. Consider closer monitoring or alternative if possible.",\n                            drug_a=drug_id,\n                            drug_a_name=drug_info.name,\n                            condition_triggered="Pregnancy",\n                            rule_type="pregnancy_warn"\n                        )\n\n        # Convert dict values to list and sort by severity\n        sorted_results = sorted(list(results.values()))\n        return sorted_results''')


In [16]:
import os

# Ensure core directory exists (already done by previous steps, but good to be explicit)
os.makedirs('core', exist_ok=True)

# Write the content of patient.py to the file
with open('core/patient.py', 'w') as f:
    f.write('''import dataclasses
from typing import List, Optional

@dataclasses.dataclass(frozen=True)
class PatientContext:
    age: int
    sex: str  # 'M' or 'F'
    egfr: Optional[float] = None  # Estimated Glomerular Filtration Rate
    conditions: List[str] = dataclasses.field(default_factory=list)
    current_meds: List[str] = dataclasses.field(default_factory=list)
    new_meds: List[str] = dataclasses.field(default_factory=list)

    @property
    def all_meds(self) -> List[str]:
        """Returns a combined list of all current and new medications."""
        return sorted(list(set(self.current_meds + self.new_meds)))''')

In [ ]:
import argparse
import sys
from core.patient import PatientContext
from core.engine import MedRecEngine, CheckResult

def main():
    parser = argparse.ArgumentParser(description="MedRec: Medication Reconciliation & Drug Interaction Checker CLI")
    parser.add_argument("--age", type=int, required=True, help="Patient's age")
    parser.add_argument("--sex", type=str, choices=['M', 'F'], required=True, help="Patient's sex (M/F)")
    parser.add_argument("--egfr", type=float, help="Patient's eGFR value")
    parser.add_argument("--conditions", type=str, help="Comma-separated active conditions (e.g., CKD,Pregnancy)")
    parser.add_argument("--current", type=str, help="Comma-separated current medications (drug_id's)")
    parser.add_argument("--new", type=str, help="Comma-separated new medications to check (drug_id's)")

    args = parser.parse_args()

    conditions_list = [c.strip() for c in args.conditions.split(',')] if args.conditions else []
    current_meds_list = [m.strip() for m in args.current.split(',')] if args.current else []
    new_meds_list = [m.strip() for m in args.new.split(',')] if args.new else []

    patient_context = PatientContext(
        age=args.age,
        sex=args.sex,
        egfr=args.egfr,
        conditions=conditions_list,
        current_meds=current_meds_list,
        new_meds=new_meds_list
    )

    engine = MedRecEngine()
    results = engine.check(patient_context)

    print(f"MedRec Check for Patient (Age: {args.age}, Sex: {args.sex}, eGFR: {args.egfr}, Conditions: {conditions_list})")
    print(f"Current Meds: {current_meds_list}")
    print(f"New Meds: {new_meds_list}")
    print("\n--- Results ---")

    if not results:
        print("No significant interactions or contraindications found.")
    else:
        for res in results:
            color = "\033[91m" if res.severity == "critical" else ("\033[93m" if res.severity == "warn" else "\033[0m") # Red/Yellow/None
            print(f"{color}{res.severity.upper()}: {res.title}\033[0m")
            print(f"  Detail: {res.detail}")
            print(f"  Action: {res.action}")
            if res.references:
                print(f"  References: {res.references}")
            print("\n")

if __name__ == '__main__':
    # In a Colab notebook, direct execution of argparse can conflict with notebook environment.
    # We'll simulate argv for demonstration purposes.
    # To run this in a standalone Python script, simply 'python cli.py --args'
    # For Colab, manually set sys.argv or call main() with a dummy parser/args.

    # Example usage for Colab: simulate command line arguments
    sys.argv = ['cli.py', '--age', '68', '--sex', 'F', '--egfr', '28', '--conditions', 'CKD,Diabetes (T2DM)', '--current', 'metformin,enalapril', '--new', 'ibuprofen,glibenclamide']
    main()
    # Or for a simple test:
    # sys.argv = ['cli.py', '--age', '70', '--sex', 'M', '--current', 'metformin', '--egfr', '25', '--conditions', 'CKD']
    # main()

    # To prevent argparse from raising SystemExit in Colab when running the cell directly without arguments:
    # if 'ipykernel' in sys.modules and hasattr(sys, 'argv'):
    #     # This block will be executed if run in a Jupyter/Colab environment
    #     print("To run the CLI in Colab, please uncomment and modify the `sys.argv` line below and then call `main()`.")
    #     print("Example: `sys.argv = ['cli.py', '--age', '68', '--sex', 'F', '--egfr', '28', '--conditions', 'CKD,Diabetes (T2DM)', '--current', 'metformin,enalapril', '--new', 'ibuprofen,glibenclamide']`")
    #     print("Then call `main()`")
    # else:
    #     # This block will be executed if run as a standalone Python script
    #     main()


MedRec Check for Patient (Age: 68, Sex: F, eGFR: 28.0, Conditions: ['CKD', 'Diabetes (T2DM)'])
Current Meds: ['metformin', 'enalapril']
New Meds: ['ibuprofen', 'glibenclamide']

--- Results ---
CRITICAL: Glibenclamide contraindicated in CKD
  Detail: Glibenclamide has active metabolites renally excreted, leading to prolonged severe hypoglycaemia in CKD patients.
  Action: Stop glibenclamide. Switch to an alternative antidiabetic agent such as gliclazide (with caution) or insulin.
  References: KDIGO 2022


CRITICAL: Metformin contraindicated: eGFR < 30
  Detail: Metformin accumulates in severe renal impairment, significantly increasing the risk of lactic acidosis, which can be fatal.
  Action: Stop metformin immediately. Switch to insulin or gliclazide (with caution).
  References: KDIGO 2022


CRITICAL: Glibenclamide contraindicated: eGFR < 30
  Detail: Glibenclamide is contraindicated for eGFR below 30. Risk of accumulation and toxicity.
  Action: Stop Glibenclamide. Consider alterna

To add a new interaction rule to data/interactions.py, you need to append a new InteractionRule instance to the RULES list. Here's a breakdown of the process and the parameters you'll need to define:

Understand the InteractionRule Dataclass: This dataclass defines the structure of each rule:

drug_a (str): The primary drug involved in the interaction (e.g., "warfarin"). Use its lowercase identifier.

drug_b (str | None): The secondary drug involved in a drug-drug interaction. Set to None if it's a condition-based rule or renal threshold rule.

condition (str | None): A patient condition that triggers the rule (e.g., "CKD", "Pregnancy"). Set to None if it's a pure drug-drug or renal threshold rule.

egfr_below (float | None): An eGFR threshold (e.g., 30.0) below which the rule applies. Set to None if eGFR is not a factor.
severity (str): The impact level of the interaction. Choose from "critical", "warn", or "info".

title (str): A concise summary of the interaction.
detail (str): A more comprehensive explanation of why the interaction occurs and its clinical significance.
action (str): Specific clinical recommendations or interventions for managing the interaction.
references (str | None): Sources or guidelines supporting the rule (e.g., "BNF", "KDIGO 2022").
Choose the Rule Type: Based on the context, decide which type of interaction you are defining:

Type A: Drug-Drug Interaction (drug_a and drug_b are set).
Type B: Condition-Drug Rule (drug_a and condition are set, drug_b is None).
Type C: Renal Threshold Rule (drug_a and egfr_below are set, drug_b and condition are None).

Type D: Combined Rule (multiple parameters like drug_a, drug_b, condition, egfr_below are set).

Construct the InteractionRule Object: Create an instance of the InteractionRule dataclass with all the relevant details for your new rule. Ensure drug names are lowercase identifiers as used in the DRUGS dictionary.

Append to the RULES List: Add your newly created InteractionRule object to the RULES list in data/interactions.py.

Here's an example of how you might add a new rule for a drug-drug interaction with a specific eGFR threshold:

RULES = [
    # ... existing rules ...

    InteractionRule(
        drug_a="digoxin",
        drug_b="amiodarone",
        condition=None,
        egfr_below=None,
        severity="critical",
        title="Digoxin + Amiodarone — increased digoxin levels",
        detail="Amiodarone can significantly increase digoxin plasma concentrations by inhibiting its renal and non-renal clearance, leading to digoxin toxicity.",
        action="Reduce digoxin dose by 30-50% and monitor digoxin levels closely when initiating amiodarone. Monitor for signs of toxicity.",
        references="BNF; American Heart Association Guidelines",
    ),
    InteractionRule(
        drug_a="cephalexin",
        drug_b=None,
        condition=None,
        egfr_below=30,
        severity="warn",
        title="Cephalexin caution: eGFR < 30",
        detail="Dose adjustment for cephalexin is usually required in patients with severe renal impairment (eGFR < 30 mL/min/1.73m2) to prevent accumulation and potential toxicity.",
        action="Reduce dose of cephalexin. Monitor renal function and clinical response. Consider alternative antibiotics if severe renal impairment is present.",
        references="IDSA Guidelines"
    )
]
Remember to keep the drug identifiers consistent (lowercase, as defined in data/drugs.py) and ensure any conditions you reference are present in the CONDITIONS set for consistency.



In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mghobashy/drug-drug-interactions")

print("Path to dataset files:", path)

100%|██████████| 1.83M/1.83M [00:00<00:00, 121MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mghobashy/drug-drug-interactions/versions/1


In [2]:
import os

# List the contents of the downloaded directory
print(f"Contents of the dataset directory: {os.listdir(path)}")

Contents of the dataset directory: ['db_drug_interactions.csv']


Now that we see the files in the dataset, the next step would be to load these files (likely CSVs or similar) into pandas DataFrames. Then, you would parse this data to create `DrugInfo` objects for the `DRUGS` dictionary and `InteractionRule` objects for the `RULES` list.

This process effectively creates your local, offline drug database within the existing framework, allowing your app to validate and generate recommendations based on this data. You can refer back to the explanations in `data/drugs.py` and `data/interactions.py` to understand the required structure for `DrugInfo` and `InteractionRule` objects.

In [3]:
import pandas as pd

# Construct the full path to the CSV file
csv_path = os.path.join(path, 'db_drug_interactions.csv')

# Load the CSV file into a pandas DataFrame
drug_interactions_df = pd.read_csv(csv_path)

# Display the first few rows of the DataFrame
print(drug_interactions_df.head())

                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description  
0  Trioxsalen may increase the photosensitizing a...  
1  Aminolevulinic acid may increase the photosens...  
2  Titanium dioxide may increase the photosensiti...  
3  Tiaprofenic acid may increase the photosensiti...  
4  Cyamemazine may increase the photosensitizing ...  


In [10]:
import sys
import os

# Add the current directory to sys.path to allow importing local modules
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Current working directory added to sys.path: {os.getcwd()}")


Current working directory added to sys.path: /content


In [11]:
from data.drugs import DRUGS, DrugInfo, CATEGORY_ANTIINFECTIVE

# Extract unique drug names from the DataFrame
drug_names_from_df = set(drug_interactions_df['Drug 1'].str.lower()).union(set(drug_interactions_df['Drug 2'].str.lower()))

# Add new drugs to the DRUGS dictionary if they don't already exist
new_drugs_added_count = 0
for drug_name in drug_names_from_df:
    if drug_name not in DRUGS:
        # Create a placeholder DrugInfo object for new drugs
        DRUGS[drug_name] = DrugInfo(
            name=drug_name.capitalize(), # Capitalize for display, but keep key lowercase
            category="Unknown", # Placeholder category
            dose_range="Unknown",
            renal_ok=None,
            renal_ci=None,
            pregnancy="unknown",
            notes="Added from external dataset. Details unknown."
        )
        new_drugs_added_count += 1

print(f"Added {new_drugs_added_count} new drugs to the DRUGS dictionary.")
print(f"Total drugs in system: {len(DRUGS)}")

Added 1692 new drugs to the DRUGS dictionary.
Total drugs in system: 1703


In [13]:
from data.interactions import RULES, InteractionRule

# Keep track of the initial number of rules
initial_rules_count = len(RULES)

# Iterate through the DataFrame and create InteractionRule objects
for index, row in drug_interactions_df.iterrows():
    drug1_lower = row['Drug 1'].lower()
    drug2_lower = row['Drug 2'].lower()
    description = row['Interaction Description']

    # Create a new InteractionRule
    new_rule = InteractionRule(
        drug_a=drug1_lower,
        drug_b=drug2_lower,
        condition=None, # No condition specified in this dataset
        egfr_below=None, # No eGFR threshold specified in this dataset
        severity="info", # Default severity, as not specified in dataset
        title=f"Interaction: {row['Drug 1']} + {row['Drug 2']}", # Generic title
        detail=description,
        action="Review patient context and consider alternatives.", # Generic action
        references="Kaggle Drug-Drug Interactions Dataset" # Reference to source
    )
    RULES.append(new_rule)

print(f"Added {len(RULES) - initial_rules_count} new interaction rules to the RULES list.")
print(f"Total rules in system: {len(RULES)}")

Added 191541 new interaction rules to the RULES list.
Total rules in system: 383090


In [14]:
from data.interactions import RULES, InteractionRule

# Keep track of the initial number of rules
initial_rules_count = len(RULES)

# Iterate through the DataFrame and create InteractionRule objects
for index, row in drug_interactions_df.iterrows():
    drug1_lower = row['Drug 1'].lower()
    drug2_lower = row['Drug 2'].lower()
    description = row['Interaction Description']

    # Create a new InteractionRule
    new_rule = InteractionRule(
        drug_a=drug1_lower,
        drug_b=drug2_lower,
        condition=None, # No condition specified in this dataset
        egfr_below=None, # No eGFR threshold specified in this dataset
        severity="info", # Default severity, as not specified in dataset
        title=f"Interaction: {row['Drug 1']} + {row['Drug 2']}", # Generic title
        detail=description,
        action="Review patient context and consider alternatives.", # Generic action
        references="Kaggle Drug-Drug Interactions Dataset" # Reference to source
    )
    RULES.append(new_rule)

print(f"Added {len(RULES) - initial_rules_count} new interaction rules to the RULES list.")
print(f"Total rules in system: {len(RULES)}")

Added 191541 new interaction rules to the RULES list.
Total rules in system: 574631


In [12]:
from data.interactions import RULES, InteractionRule

# Keep track of the initial number of rules
initial_rules_count = len(RULES)

# Iterate through the DataFrame and create InteractionRule objects
for index, row in drug_interactions_df.iterrows():
    drug1_lower = row['Drug 1'].lower()
    drug2_lower = row['Drug 2'].lower()
    description = row['Interaction Description']

    # Create a new InteractionRule
    new_rule = InteractionRule(
        drug_a=drug1_lower,
        drug_b=drug2_lower,
        condition=None, # No condition specified in this dataset
        egfr_below=None, # No eGFR threshold specified in this dataset
        severity="info", # Default severity, as not specified in dataset
        title=f"Interaction: {row['Drug 1']} + {row['Drug 2']}", # Generic title
        detail=description,
        action="Review patient context and consider alternatives.", # Generic action
        references="Kaggle Drug-Drug Interactions Dataset" # Reference to source
    )
    RULES.append(new_rule)

print(f"Added {len(RULES) - initial_rules_count} new interaction rules to the RULES list.")
print(f"Total rules in system: {len(RULES)}")

Added 191541 new interaction rules to the RULES list.
Total rules in system: 191549


In [23]:
import dataclasses
from typing import List, Optional

# 1. Update Drug Data Structure to include Drug Class
@dataclasses.dataclass
class DrugInfo:
    name: str
    category: str
    drug_class: str  # e.g., "ACEi", "Beta-Blocker", "NSAID"
    dose_range: str
    renal_ok: Optional[float] = None
    renal_ci: Optional[float] = None
    pregnancy: Optional[str] = None
    notes: Optional[str] = None

# Populate DRUGS with drug_class mapping
DRUGS = {
    "enalapril": DrugInfo(
        name="Enalapril",
        category="Cardiovascular",
        drug_class="ACEi",
        dose_range="5-40 mg/day",
        notes="ACE Inhibitor"
    ),
    "carvedilol": DrugInfo(
        name="Carvedilol",
        category="Cardiovascular",
        drug_class="Beta-Blocker",
        dose_range="3.125-25 mg 2x/day"
    ),
    "ibuprofen": DrugInfo(
        name="Ibuprofen",
        category="Analgesic",
        drug_class="NSAID",
        dose_range="200-800 mg 3x/day"
    )
}

# 2. Update Interaction Rule to support Drug Names OR Drug Classes
@dataclasses.dataclass
class InteractionRule:
    drug_a: Optional[str] = None          # Specific drug name (e.g., "enalapril")
    drug_class_a: Optional[str] = None    # Drug class (e.g., "ACEi")
    drug_b: Optional[str] = None
    drug_class_b: Optional[str] = None
    condition: Optional[str] = None       # Medical condition (e.g., "Asthma/COPD")
    egfr_below: Optional[float] = None
    severity: str = "warn"                # "critical" | "warn" | "info"
    title: str = ""
    detail: str = ""
    action: str = ""
    references: Optional[str] = None

# Class-level rules list
RULES = [
    # Class-to-Condition Rule: ACEi + Asthma Caution Rule
    InteractionRule(
        drug_class_a="ACEi",
        condition="Asthma/COPD",
        severity="warn",
        title="ACE Inhibitor Caution in Asthma/COPD",
        detail="ACE inhibitors increase bradykinin levels, which can trigger or worsen chronic cough and bronchial hyperreactivity in asthma patients.",
        action="Use with caution. Monitor for persistent cough or bronchospasm. Consider ARB if cough develops.",
        references="BNF / Clinical Guidance"
    ),
    # Class-to-Class Interaction Rule: ACEi + NSAID
    InteractionRule(
        drug_class_a="ACEi",
        drug_class_b="NSAID",
        severity="warn",
        title="ACEi + NSAID Renal Risk",
        detail="Combining an ACE inhibitor with an NSAID compromises renal autoregulation and increases the risk of acute kidney injury.",
        action="Avoid chronic co-administration. Monitor renal function and blood pressure if necessary.",
        references="EMA Guidelines"
    )
]

# 3. Updated Validation Engine
class MedRecEngine:
    def validate(self, patient) -> List[InteractionRule]:
        detected_alerts = []
        patient_meds = [m.lower() for m in patient.current_meds + patient.new_meds]

        # Extract drug classes for all active patient medications
        patient_classes = set()
        for med in patient_meds:
            if med in DRUGS and DRUGS[med].drug_class:
                patient_classes.add(DRUGS[med].drug_class.upper())

        for rule in RULES:
            # Match specific drug OR drug class
            has_a = (rule.drug_a and rule.drug_a.lower() in patient_meds) or \
                    (rule.drug_class_a and rule.drug_class_a.upper() in patient_classes)

            has_b = (rule.drug_b and rule.drug_b.lower() in patient_meds) or \
                    (rule.drug_class_b and rule.drug_class_b.upper() in patient_classes)

            # Check Drug-to-Drug or Class-to-Class Interaction
            if has_a and has_b:
                detected_alerts.append(rule)
                continue

            # Check Condition Contraindications
            if has_a and rule.condition and rule.condition in patient.conditions:
                detected_alerts.append(rule)
                continue

            # Check Renal Thresholds
            if has_a and rule.egfr_below is not None and patient.egfr < rule.egfr_below:
                detected_alerts.append(rule)

        return detected_alerts

In [24]:
import os
import dataclasses
import pandas as pd
import gradio as gr
import kagglehub

# ==========================================
# 1. CORE DATA STRUCTURES & DATACLASSES
# ==========================================

@dataclasses.dataclass
class DrugInfo:
    name: str
    category: str
    dose_range: str
    renal_ok: float | None = None
    renal_ci: float | None = None
    pregnancy: str | None = None
    notes: str | None = None

@dataclasses.dataclass
class InteractionRule:
    drug_a: str
    drug_b: str | None
    condition: str | None
    egfr_below: float | None
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    references: str | None = None

@dataclasses.dataclass
class PatientContext:
    age: int
    sex: str
    egfr: float
    conditions: list[str]
    current_meds: list[str]
    new_meds: list[str]

# Global Storage (will be populated from dataset)
DRUGS: dict[str, DrugInfo] = {}
RULES: list[InteractionRule] = []

# Define conditions for type-checking and consistency (moved here for self-contained app)
CONDITIONS = {
    "CKD",
    "Diabetes (T2DM)",
    "Pregnancy",
    "Asthma/COPD",
    "Heart Failure"
}

# ==========================================
# 2. DATA LOADING & DYNAMIC POPULATION
# ==========================================

# Download latest version of the Kaggle dataset (moved here for self-contained app)
path_to_dataset = kagglehub.dataset_download("mghobashy/drug-drug-interactions")

# Construct the full path to the CSV file
csv_path = os.path.join(path_to_dataset, 'db_drug_interactions.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Clean column headers
    df.columns = df.columns.str.strip()

    # Populate DRUGS dictionary with unique entities from dataset
    unique_drugs = set(df['Drug 1'].str.lower()).union(set(df['Drug 2'].str.lower()))
    for drug in unique_drugs:
        if drug not in DRUGS:
            DRUGS[drug] = DrugInfo(
                name=drug.capitalize(),
                category="General",
                dose_range="Standard",
                notes="Imported from dataset"
            )

    # Populate RULES list from dataset rows
    for _, row in df.iterrows():
        d1 = str(row['Drug 1']).strip().lower()
        d2 = str(row['Drug 2']).strip().lower()
        desc = str(row['Interaction Description']).strip()

        RULES.append(
            InteractionRule(
                drug_a=d1,
                drug_b=d2,
                condition=None,
                egfr_below=None,
                severity="warn",
                title=f"Interaction: {d1.capitalize()} + {d2.capitalize()}",
                detail=desc,
                action="Monitor patient closely or consider alternative therapies.",
                references="Kaggle DDI Dataset"
            )
        )


# ==========================================_
# 3. RECONCILIATION ENGINE
# ==========================================_

class MedRecEngine:
    def validate(self, patient: PatientContext) -> list[InteractionRule]:
        detected_alerts = []
        all_meds = [m.lower() for m in patient.current_meds + patient.new_meds]

        # Check interaction rules
        for rule in RULES:
            # Drug-Drug Interaction Check
            if rule.drug_a in all_meds and rule.drug_b in all_meds:
                detected_alerts.append(rule)
                continue

            # Renal Safety Check
            if rule.drug_a in all_meds and rule.egfr_below is not None:
                if patient.egfr < rule.egfr_below:
                    detected_alerts.append(rule)
                    continue

            # Condition Contraindication Check
            if rule.drug_a in all_meds and rule.condition in patient.conditions:
                detected_alerts.append(rule)

        return detected_alerts

engine = MedRecEngine()

# ==========================================_
# 4. GRADIO INTERACTIVE INTERFACE
# ==========================================_

def run_reconciliation(age, sex, egfr, conditions, current_meds, new_meds):
    # conditions, current_meds, new_meds are now lists directly from the dropdowns
    current_meds_lower = [m.lower() for m in current_meds]
    new_meds_lower = [m.lower() for m in new_meds]

    patient = PatientContext(
        age=int(age),
        sex=sex,
        egfr=float(egfr),
        conditions=conditions,
        current_meds=current_meds_lower,
        new_meds=new_meds_lower,
    )

    alerts = engine.validate(patient)

    if not alerts:
        return "### ✅ No critical interactions found for this patient context."

    output = f"### ⚠️ Flagged Alerts ({len(alerts)})\n\n"
    for alert in alerts:
        severity_icon = "🔴" if alert.severity == "critical" else ("🟡" if alert.severity == "warn" else "ℹ️") # Added info icon
        output += f"#### {severity_icon} [{alert.severity.upper()}] {alert.title}\n"
        output += f"**Details:** {alert.detail}\n\n"
        output += f"**Action:** {alert.action}\n\n"
        if alert.references:
            output += f"*Source: {alert.references}*\n"
        output += "---\n"

    return output

# Prepare choices for dropdowns
all_drug_names = sorted([drug_info.name for drug_info in DRUGS.values()])
all_conditions = sorted(list(CONDITIONS))

with gr.Blocks(title="MedRec Engine") as demo:
    gr.Markdown("# 💊 Medication Reconciliation & Safety Checker")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Patient Parameters")
            age = gr.Number(label="Age", value=65)
            sex = gr.Dropdown(choices=["Male", "Female", "Other"], label="Sex", value="Male")
            egfr = gr.Number(label="eGFR (mL/min/1.73m²)", value=28)
            conditions = gr.Dropdown(
                choices=all_conditions,
                multiselect=True,
                label="Conditions",
                value=["CKD", "Diabetes (T2DM)"] # Example initial values
            )

            gr.Markdown("### Regimen")
            current_meds = gr.Dropdown(
                choices=all_drug_names,
                multiselect=True,
                label="Current Medications",
                value=["Cyamemazine"] # Updated initial value from Kaggle dataset
            )
            new_meds = gr.Dropdown(
                choices=all_drug_names,
                multiselect=True,
                label="New Proposed Medications",
                value=["Verteporfin"] # Updated initial value from Kaggle dataset
            )

            btn = gr.Button("Run Safety Validation", variant="primary")

        with gr.Column():
            gr.Markdown("### Clinical Recommendations")
            results_output = gr.Markdown()

    btn.click(
        fn=run_reconciliation,
        inputs=[age, sex, egfr, conditions, current_meds, new_meds],
        outputs=results_output
    )

if __name__ == "__main__":
    demo.launch(inline=True, share=False)

Using Colab cache for faster access to the 'drug-drug-interactions' dataset.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [25]:
import gradio as gr

# 1. Kill any active Gradio app running on local ports
gr.close_all()

# 2. Re-launch with debug mode enabled and a public share link generated
demo.launch(inline=False, share=True, debug=True)

Closing server running on port: 7860
Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://15546d0102ec871e3c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://15546d0102ec871e3c.gradio.live


In [28]:
!pip install streamlit
import streamlit as st
# (Include PatientContext, DRUGS, RULES, and MedRecEngine logic from above)

st.title("💊 Medication Reconciliation Engine")

age = st.number_input("Age", value=65)
sex = st.selectbox("Sex", ["Male", "Female", "Other"])
egfr = st.number_input("eGFR", value=28.0)
conditions = st.text_input("Conditions (comma-separated)", "Asthma, T2DM")
current_meds = st.text_input("Current Meds", "enalapril")
new_meds = st.text_input("New Meds", "spironolactone")

if st.button("Run Safety Validation"):
    patient = PatientContext(
        age=age, sex=sex, egfr=egfr,
        conditions=[c.strip() for c in conditions.split(",") if c],
        current_meds=[m.strip() for m in current_meds.split(",") if m],
        new_meds=[m.strip() for m in new_meds.split(",") if m]
    )

    alerts = engine.validate(patient)
    if not alerts:
        st.success("No interactions detected.")
    else:
        for a in alerts:
            st.warning(f"**[{a.severity.upper()}] {a.title}**\n\n{a.detail}\n\n*Action:* {a.action}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 98.8 MB/s eta 0:00:00


2026-08-22 19:38:40.197 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.324 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-22 19:38:40.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.327 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.329 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.332 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 19:38:40.335 Session state does not 

In [31]:
# Create app.py
!app.py

/bin/bash: line 1: app.py: command not found


In [30]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


First, let's create the `app.py` file with your Streamlit application code. I will include the necessary data structures and engine logic directly within this file for simplicity, ensuring it's self-contained.

In [32]:
import os
import dataclasses
from typing import List, Optional, Set, Dict

# ==========================================
# 1. CORE DATA STRUCTURES & DATACLASSES
# ==========================================

@dataclasses.dataclass
class DrugInfo:
    name: str
    category: str
    dose_range: str
    renal_ok: float | None = None
    renal_ci: float | None = None
    pregnancy: str | None = None
    notes: str | None = None

@dataclasses.dataclass(frozen=True)
class InteractionRule:
    drug_a: str
    drug_b: str | None
    condition: str | None
    egfr_below: float | None
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    references: str | None = None

    def __lt__(self, other): # For sorting results by severity
        severity_order = {"critical": 0, "warn": 1, "info": 2}
        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)

@dataclasses.dataclass(frozen=True)
class CheckResult:
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    drug_a: str
    drug_a_name: str
    drug_b: Optional[str] = None
    drug_b_name: Optional[str] = None
    condition_triggered: Optional[str] = None
    egfr_triggered: Optional[float] = None
    rule_type: str = "interaction" # 'interaction', 'renal_ci', 'renal_ok', 'pregnancy_ci', 'pregnancy_warn'
    references: Optional[str] = None

    def __lt__(self, other): # For sorting results by severity
        severity_order = {"critical": 0, "warn": 1, "info": 2}
        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)


@dataclasses.dataclass(frozen=True)
class PatientContext:
    age: int
    sex: str  # 'M' or 'F'
    egfr: Optional[float] = None  # Estimated Glomerular Filtration Rate
    conditions: List[str] = dataclasses.field(default_factory=list)
    current_meds: List[str] = dataclasses.field(default_factory=list)
    new_meds: List[str] = dataclasses.field(default_factory=list)

    @property
    def all_meds(self) -> List[str]:
        """Returns a combined list of all current and new medications."""
        return sorted(list(set(self.current_meds + self.new_meds)))


# Global Storage (will be populated from dataset)
DRUGS: dict[str, DrugInfo] = {}
RULES: list[InteractionRule] = []

# Define conditions for type-checking and consistency (moved here for self-contained app)
CONDITIONS = {
    "CKD",
    "Diabetes (T2DM)",
    "Pregnancy",
    "Asthma/COPD",
    "Heart Failure"
}

# ==========================================
# 2. DATA LOADING & DYNAMIC POPULATION (Simplified for app.py)
# ==========================================

# Populate DRUGS dictionary (using the existing DRUGS from the notebook state)
# In a real app, this would be loaded from a persistent source.
# For this demonstration, we'll assume DRUGS and RULES are already populated
# in the Colab environment from previous cells. If not, you'd load them here.

# Since DRUGS and RULES are already populated from previous notebook cells
# (e.g., cell 1bdebed9 and f3528740), we will rely on those.
# However, for a truly self-contained Streamlit app.py, you would put the
# data loading logic directly here. For demonstration, we will assume these
# globals are available from the notebook's execution context.

# Placeholder for a truly self-contained app:
# import pandas as pd
# import kagglehub
# path_to_dataset = kagglehub.dataset_download("mghobashy/drug-drug-interactions")
# csv_path = os.path.join(path_to_dataset, 'db_drug_interactions.csv')
# df = pd.read_csv(csv_path)
# ... (logic to populate DRUGS and RULES from df)


# ==========================================
# 3. RECONCILIATION ENGINE
# ==========================================

class MedRecEngine:
    def check(self, ctx: PatientContext) -> List[CheckResult]:
        results: Dict[str, CheckResult] = {}

        all_meds_ids = ctx.all_meds
        all_meds_info = {drug_id: DRUGS[drug_id] for drug_id in all_meds_ids if drug_id in DRUGS}

        # --- Pass 1: Rule-based interaction check ---
        for rule in RULES:
            if rule.drug_a not in all_meds_ids:
                continue

            drug_a_info = DRUGS.get(rule.drug_a)
            if not drug_a_info:
                continue # Should not happen if data is clean

            # Check drug_b presence if required
            if rule.drug_b and rule.drug_b not in all_meds_ids:
                continue
            drug_b_info = DRUGS.get(rule.drug_b) if rule.drug_b else None

            # Check condition if required
            if rule.condition and rule.condition not in ctx.conditions:
                continue

            # Check eGFR if required
            if rule.egfr_below is not None:
                if ctx.egfr is None or ctx.egfr >= rule.egfr_below:
                    continue

            # If all conditions met, add result (deduplicated by title)
            if rule.title not in results:
                results[rule.title] = CheckResult(
                    severity=rule.severity,
                    title=rule.title,
                    detail=rule.detail,
                    action=rule.action,
                    drug_a=rule.drug_a,
                    drug_a_name=drug_a_info.name,
                    drug_b=rule.drug_b,
                    drug_b_name=drug_b_info.name if drug_b_info else None,
                    condition_triggered=rule.condition,
                    egfr_triggered=rule.egfr_below,
                    rule_type="interaction",
                    references=rule.references
                )

        # --- Pass 2: Per-drug renal safety backstop ---
        if ctx.egfr is not None:
            for drug_id, drug_info in all_meds_info.items():
                # Prioritize explicit rules, skip if a rule with the same title already fired
                # This is a simplification; a more robust approach might be needed for complex overlaps

                if drug_info.renal_ci is not None and ctx.egfr < drug_info.renal_ci:
                    title = f"{drug_info.name} contraindicated: eGFR < {int(drug_info.renal_ci)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated for eGFR below {int(drug_info.renal_ci)}. Risk of accumulation and toxicity.",
                            action=f"Stop {drug_info.name}. Consider alternative therapy.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ci,
                            rule_type="renal_ci"
                        )

                elif drug_info.renal_ok is not None and ctx.egfr < drug_info.renal_ok:
                    title = f"{drug_info.name} caution: eGFR < {int(drug_info.renal_ok)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution for eGFR below {int(drug_info.renal_ok)}. Consider dose reduction or increased monitoring.",
                            action=f"Monitor renal function and drug levels. Consider dose adjustment.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ok,
                            rule_type="renal_ok"
                        )

        # --- Pass 3: Per-drug pregnancy safety ---
        if "Pregnancy" in ctx.conditions:
            for drug_id, drug_info in all_meds_info.items():
                # Explicit rules take priority for pregnancy too
                # Check if any rule-based result for this drug already covers pregnancy
                # This is a simplification, more specific checking might be needed

                if drug_info.pregnancy == "contraindicated":
                    title = f"{drug_info.name} contraindicated in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated during pregnancy due to potential foetal harm.",
                            action=f"Stop {drug_info.name} immediately. Discuss alternatives with patient.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_ci"
                        )
                elif drug_info.pregnancy == "caution":
                    title = f"{drug_info.name} caution in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution during pregnancy. Weigh benefits against potential risks.",
                            action=f"Review necessity of {drug_info.name}. Consider closer monitoring or alternative if possible.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_warn"
                        )

        # Convert dict values to list and sort by severity
        sorted_results = sorted(list(results.values()))
        return sorted_results

engine = MedRecEngine()


# ==========================================
# 4. STREAMLIT INTERFACE
# ==========================================
import streamlit as st

st.set_page_config(layout="wide")
st.title("💊 Medication Reconciliation & Safety Checker")

# Collect available drugs from the DRUGS global variable
# Ensure DRUGS is populated from the notebook's global state
available_drugs = sorted([drug_info.name for drug_info in DRUGS.values()]) if DRUGS else []
available_conditions = sorted(list(CONDITIONS)) if CONDITIONS else []

with st.sidebar:
    st.header("Patient Parameters")
    age = st.number_input("Age", min_value=0, max_value=120, value=65)
    sex = st.selectbox("Sex", ["Male", "Female", "Other"], index=0)
    egfr = st.number_input("eGFR (mL/min/1.73m²)", min_value=0.0, max_value=200.0, value=28.0)
    conditions = st.multiselect(
        "Conditions",
        options=available_conditions,
        default=["CKD", "Diabetes (T2DM)"] if "CKD" in available_conditions and "Diabetes (T2DM)" in available_conditions else []
    )

    st.header("Medication Regimen")
    current_meds_selected = st.multiselect(
        "Current Medications",
        options=available_drugs,
        default=["Metformin", "Enalapril"] if "Metformin" in available_drugs and "Enalapril" in available_drugs else []
    )
    new_meds_selected = st.multiselect(
        "New Proposed Medications",
        options=available_drugs,
        default=["Ibuprofen", "Glibenclamide"] if "Ibuprofen" in available_drugs and "Glibenclamide" in available_drugs else []
    )

    run_button = st.button("Run Safety Validation", type="primary")


if run_button:
    # Convert selected drug names back to lowercase IDs for the engine
    current_med_ids = [name.lower() for name in current_meds_selected]
    new_med_ids = [name.lower() for name in new_meds_selected]

    patient_context = PatientContext(
        age=age,
        sex=sex,
        egfr=egfr,
        conditions=conditions,
        current_meds=current_med_ids,
        new_meds=new_med_ids
    )

    alerts = engine.check(patient_context)

    st.header("Clinical Recommendations")

    if not alerts:
        st.success("### ✅ No critical interactions found for this patient context.")
    else:
        st.warning(f"### ⚠️ Flagged Alerts ({len(alerts)})")
        for alert in alerts:
            if alert.severity == "critical":
                st.error(f"**🔴 [CRITICAL] {alert.title}**")
            elif alert.severity == "warn":
                st.warning(f"**🟡 [WARNING] {alert.title}**")
            else:
                st.info(f"**ℹ️ [INFO] {alert.title}**")

            st.write(f"**Details:** {alert.detail}")
            st.write(f"**Action:** {alert.action}")
            if alert.references:
                st.caption(f"*Source: {alert.references}*")
            st.markdown("--- ")



2026-08-22 20:07:55.372 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.376 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.377 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.378 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.380 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.382 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 20:07:55.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [33]:
# Write the Streamlit app code to a file named app.py
with open('app.py', 'w') as f:
    f.write('''
import os
import dataclasses
from typing import List, Optional, Set, Dict
import streamlit as st


# ==========================================
# 1. CORE DATA STRUCTURES & DATACLASSES
# ==========================================

@dataclasses.dataclass
class DrugInfo:
    name: str
    category: str
    dose_range: str
    renal_ok: float | None = None
    renal_ci: float | None = None
    pregnancy: str | None = None
    notes: str | None = None

@dataclasses.dataclass(frozen=True)
class InteractionRule:
    drug_a: str
    drug_b: str | None
    condition: str | None
    egfr_below: float | None
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    references: str | None = None

    def __lt__(self, other): # For sorting results by severity
        severity_order = {"critical": 0, "warn": 1, "info": 2}
        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)

@dataclasses.dataclass(frozen=True)
class CheckResult:
    severity: str  # "critical" | "warn" | "info"
    title: str
    detail: str
    action: str
    drug_a: str
    drug_a_name: str
    drug_b: Optional[str] = None
    drug_b_name: Optional[str] = None
    condition_triggered: Optional[str] = None
    egfr_triggered: Optional[float] = None
    rule_type: str = "interaction" # 'interaction', 'renal_ci', 'renal_ok', 'pregnancy_ci', 'pregnancy_warn'
    references: Optional[str] = None

    def __lt__(self, other): # For sorting results by severity
        severity_order = {"critical": 0, "warn": 1, "info": 2}
        return severity_order.get(self.severity, 99) < severity_order.get(other.severity, 99)


@dataclasses.dataclass(frozen=True)
class PatientContext:
    age: int
    sex: str  # 'M' or 'F'
    egfr: Optional[float] = None  # Estimated Glomerular Filtration Rate
    conditions: List[str] = dataclasses.field(default_factory=list)
    current_meds: List[str] = dataclasses.field(default_factory=list)
    new_meds: List[str] = dataclasses.field(default_factory=list)

    @property
    def all_meds(self) -> List[str]:
        """Returns a combined list of all current and new medications."""
        return sorted(list(set(self.current_meds + self.new_meds)))


# Global Storage (will be populated from dataset)
DRUGS: dict[str, DrugInfo] = {}
RULES: list[InteractionRule] = []

# Define conditions for type-checking and consistency (moved here for self-contained app)
CONDITIONS = {
    "CKD",
    "Diabetes (T2DM)",
    "Pregnancy",
    "Asthma/COPD",
    "Heart Failure"
}

# ==========================================
# 2. DATA LOADING & DYNAMIC POPULATION (Simplified for app.py)
# ==========================================

# To make app.py truly self-contained, we will hardcode a minimal set of drugs and rules.
# In a real application, you would load these from persistent storage (e.g., a database, CSV, or JSON file)
# or use more sophisticated data loading from your Kaggle dataset.

# Example DRUGS (minimal set, for demonstration in a self-contained app.py)
DRUGS = {
    "metformin": DrugInfo(
        name="Metformin", category="Antidiabetic", dose_range="500-2000 mg/day",
        renal_ok=45, renal_ci=30, pregnancy="unknown", notes="First-line for T2DM."
    ),
    "enalapril": DrugInfo(
        name="Enalapril", category="Cardiovascular", dose_range="5-40 mg/day",
        renal_ok=30, renal_ci=10, pregnancy="contraindicated", notes="ACE inhibitor."
    ),
    "ibuprofen": DrugInfo(
        name="Ibuprofen", category="Anti-inflammatory", dose_range="200-800 mg 3x/day",
        renal_ok=30, renal_ci=15, pregnancy="caution", notes="NSAID."
    ),
    "glibenclamide": DrugInfo(
        name="Glibenclamide", category="Antidiabetic", dose_range="2.5-20 mg/day",
        renal_ok=50, renal_ci=30, pregnancy="caution", notes="Sulfonylurea."
    ),
    "spironolactone": DrugInfo(
        name="Spironolactone", category="Cardiovascular", dose_range="25-100 mg/day",
        renal_ok=45, renal_ci=30, pregnancy="unknown", notes="Potassium-sparing diuretic."
    )
}

# Example RULES (minimal set)
RULES = [
    InteractionRule(
        drug_a="enalapril", drug_b="spironolactone", condition=None,
        egfr_below=None, severity="warn",
        title="ACEi + Spironolactone — risk of hyperkalaemia",
        detail="Both drugs increase potassium. Combination can lead to dangerous hyperkalaemia.",
        action="Monitor serum potassium closely.", references="BNF"
    ),
    InteractionRule(
        drug_a="metformin", drug_b=None, condition=None,
        egfr_below=30, severity="critical",
        title="Metformin contraindicated: eGFR < 30",
        detail="Metformin accumulates in severe renal impairment, increasing lactic acidosis risk.",
        action="Stop metformin immediately.", references="KDIGO 2022"
    ),
    InteractionRule(
        drug_a="glibenclamide", drug_b=None, condition="CKD",
        egfr_below=None, severity="critical",
        title="Glibenclamide contraindicated in CKD",
        detail="Glibenclamide has active metabolites renally excreted, leading to prolonged severe hypoglycaemia.",
        action="Stop glibenclamide.", references="KDIGO 2022"
    ),
    InteractionRule(
        drug_a="ibuprofen", drug_b="enalapril", condition=None,
        egfr_below=60, severity="warn",
        title="Triple Whammy: NSAID + ACEi in renal impairment",
        detail="Combination of Ibuprofen (NSAID) and Enalapril (ACEi) can cause acute kidney injury.",
        action="Avoid combination, especially if eGFR < 60.", references="European Medicines Agency"
    ),
    InteractionRule(
        drug_a="enalapril", drug_b=None, condition="Pregnancy",
        egfr_below=None, severity="critical",
        title="ACE inhibitors contraindicated in pregnancy",
        detail="ACE inhibitors can cause foetal renal dysfunction.",
        action="Stop enalapril immediately.", references="FDA Black Box Warning"
    )
]



# ==========================================
# 3. RECONCILIATION ENGINE
# ==========================================

class MedRecEngine:
    def check(self, ctx: PatientContext) -> List[CheckResult]:
        results: Dict[str, CheckResult] = {}

        all_meds_ids = ctx.all_meds
        all_meds_info = {drug_id: DRUGS[drug_id] for drug_id in all_meds_ids if drug_id in DRUGS}

        # --- Pass 1: Rule-based interaction check ---
        for rule in RULES:
            if rule.drug_a not in all_meds_ids:
                continue

            drug_a_info = DRUGS.get(rule.drug_a)
            if not drug_a_info:
                continue # Should not happen if data is clean

            # Check drug_b presence if required
            if rule.drug_b and rule.drug_b not in all_meds_ids:
                continue
            drug_b_info = DRUGS.get(rule.drug_b) if rule.drug_b else None

            # Check condition if required
            if rule.condition and rule.condition not in ctx.conditions:
                continue

            # Check eGFR if required
            if rule.egfr_below is not None:
                if ctx.egfr is None or ctx.egfr >= rule.egfr_below:
                    continue

            # If all conditions met, add result (deduplicated by title)
            if rule.title not in results:
                results[rule.title] = CheckResult(
                    severity=rule.severity,
                    title=rule.title,
                    detail=rule.detail,
                    action=rule.action,
                    drug_a=rule.drug_a,
                    drug_a_name=drug_a_info.name,
                    drug_b=rule.drug_b,
                    drug_b_name=drug_b_info.name if drug_b_info else None,
                    condition_triggered=rule.condition,
                    egfr_triggered=rule.egfr_below,
                    rule_type="interaction",
                    references=rule.references
                )

        # --- Pass 2: Per-drug renal safety backstop ---
        if ctx.egfr is not None:
            for drug_id, drug_info in all_meds_info.items():
                # Prioritize explicit rules, skip if a rule with the same title already fired
                # This is a simplification; a more robust approach might be needed for complex overlaps

                if drug_info.renal_ci is not None and ctx.egfr < drug_info.renal_ci:
                    title = f"{drug_info.name} contraindicated: eGFR < {int(drug_info.renal_ci)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated for eGFR below {int(drug_info.renal_ci)}. Risk of accumulation and toxicity.",
                            action=f"Stop {drug_info.name}. Consider alternative therapy.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ci,
                            rule_type="renal_ci"
                        )

                elif drug_info.renal_ok is not None and ctx.egfr < drug_info.renal_ok:
                    title = f"{drug_info.name} caution: eGFR < {int(drug_info.renal_ok)}"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution for eGFR below {int(drug_info.renal_ok)}. Consider dose reduction or increased monitoring.",
                            action=f"Monitor renal function and drug levels. Consider dose adjustment.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            egfr_triggered=drug_info.renal_ok,
                            rule_type="renal_ok"
                        )

        # --- Pass 3: Per-drug pregnancy safety ---
        if "Pregnancy" in ctx.conditions:
            for drug_id, drug_info in all_meds_info.items():
                # Explicit rules take priority for pregnancy too
                # Check if any rule-based result for this drug already covers pregnancy
                # This is a simplification, more specific checking might be needed

                if drug_info.pregnancy == "contraindicated":
                    title = f"{drug_info.name} contraindicated in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="critical",
                            title=title,
                            detail=f"{drug_info.name} is contraindicated during pregnancy due to potential foetal harm.",
                            action=f"Stop {drug_info.name} immediately. Discuss alternatives with patient.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_ci"
                        )
                elif drug_info.pregnancy == "caution":
                    title = f"{drug_info.name} caution in pregnancy"
                    if title not in results:
                        results[title] = CheckResult(
                            severity="warn",
                            title=title,
                            detail=f"Use {drug_info.name} with caution during pregnancy. Weigh benefits against potential risks.",
                            action=f"Review necessity of {drug_info.name}. Consider closer monitoring or alternative if possible.",
                            drug_a=drug_id,
                            drug_a_name=drug_info.name,
                            condition_triggered="Pregnancy",
                            rule_type="pregnancy_warn"
                        )

        # Convert dict values to list and sort by severity
        sorted_results = sorted(list(results.values()))
        return sorted_results

engine = MedRecEngine()


# ==========================================
# 4. STREAMLIT INTERFACE
# ==========================================

st.set_page_config(layout="wide")
st.title("💊 Medication Reconciliation & Safety Checker")

# Collect available drugs from the DRUGS global variable
available_drugs = sorted([drug_info.name for drug_info in DRUGS.values()]) if DRUGS else []
available_conditions = sorted(list(CONDITIONS)) if CONDITIONS else []

with st.sidebar:
    st.header("Patient Parameters")
    age = st.number_input("Age", min_value=0, max_value=120, value=65)
    sex = st.selectbox("Sex", ["Male", "Female", "Other"], index=0)
    egfr = st.number_input("eGFR (mL/min/1.73m²)", min_value=0.0, max_value=200.0, value=28.0)
    conditions = st.multiselect(
        "Conditions",
        options=available_conditions,
        default=["CKD", "Diabetes (T2DM)"] if "CKD" in available_conditions and "Diabetes (T2DM)" in available_conditions else []
    )

    st.header("Medication Regimen")
    current_meds_selected = st.multiselect(
        "Current Medications",
        options=available_drugs,
        default=["Metformin", "Enalapril"] if "Metformin" in available_drugs and "Enalapril" in available_drugs else []
    )
    new_meds_selected = st.multiselect(
        "New Proposed Medications",
        options=available_drugs,
        default=["Ibuprofen", "Glibenclamide"] if "Ibuprofen" in available_drugs and "Glibenclamide" in available_drugs else []
    )

    run_button = st.button("Run Safety Validation", type="primary")


if run_button:
    # Convert selected drug names back to lowercase IDs for the engine
    current_med_ids = [name.lower() for name in current_meds_selected]
    new_med_ids = [name.lower() for name in new_meds_selected]

    patient_context = PatientContext(
        age=age,
        sex=sex,
        egfr=egfr,
        conditions=conditions,
        current_meds=current_med_ids,
        new_meds=new_med_ids
    )

    alerts = engine.check(patient_context)

    st.header("Clinical Recommendations")

    if not alerts:
        st.success("### ✅ No critical interactions found for this patient context.")
    else:
        st.warning(f"### ⚠️ Flagged Alerts ({len(alerts)})")
        for alert in alerts:
            if alert.severity == "critical":
                st.error(f"**🔴 [CRITICAL] {alert.title}**")
            elif alert.severity == "warn":
                st.warning(f"**🟡 [WARNING] {alert.title}**")
            else:
                st.info(f"**ℹ️ [INFO] {alert.title}**")

            st.write(f"**Details:** {alert.detail}")
            st.write(f"**Action:** {alert.action}")
            if alert.references:
                st.caption(f"*Source: {alert.references}*")
            st.markdown("--- ")
'''
)
print("Streamlit app code written to app.py")

Streamlit app code written to app.py


Next, to make the Streamlit app accessible in your browser, we'll install `ngrok` and then run the Streamlit app. `ngrok` creates a secure tunnel to your local Streamlit server, giving you a public URL.

In [34]:
!npm install localtunnel

# Run Streamlit in the background and use ngrok to expose it
# Streamlit typically runs on port 8501
get_ipython().system_raw('streamlit run app.py & npx localtunnel --port 8501')

# Give it a moment to start and print the public URL
import time
time.sleep(5)

# Attempt to capture the URL from the localtunnel output
# This might require some manual inspection if automatic capture fails
print("Your Streamlit app should be accessible at the localtunnel URL above. Look for a line like: 'your url is: https://something.loca.lt'")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦


KeyboardInterrupt



KeyboardInterrupt: 